# SOXX 原始 Yahoo 收盘价与发行人 NAV 来源对齐

固定一次来源请求，原生 Qlib 日历对齐；只做同日价格口径检查，不读取未来收益。


In [1]:
import os,sys,json,hashlib,re,datetime,math,tarfile,tempfile,subprocess,shlex
from pathlib import Path
from zoneinfo import ZoneInfo
from xml.etree import ElementTree as ET
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
root=Path('/Users/rziiiii/dev/workspace');sys.path.insert(0,str(root))
from src.data.nas_market_data import archive_nas_recorder,_settings
card=root/'research/stage1_soxx_nav_yahoo_raw_close_source_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['sources'];snap=root/'data/snapshots/stage1_soxx_nav_yahoo_raw_close_source_20261023/chart.json'
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and sha(root/src['official_NAV_raw'])==src['official_NAV_sha256'] and sha(root/src['failed_Qlib_scale_card']/'summary.json')==src['failed_Qlib_scale_summary_sha256']
assert sha(root/src['Qlib_calendar_provider']/'qlib.yaml')==src['Qlib_config_sha256'] and all(plan['budget'][k]==0 for k in ('model_fits','native_accounts','bootstraps'))
t=json.loads((card/'transport_result.json').read_text());a=t['attempt'];assert t['plan_sha256']==sha(card/'plan.json') and t['GETs_consumed']==1 and t['retries']==0
assert a['http_status']==200 and a['body_sha256']==sha(snap) and a['body_bytes']==snap.stat().st_size and a['path']==plan['exact_GET']['path'] and a['query']==plan['exact_GET']['query']
receipt=json.loads((card/'nas_source_archive.json').read_text());assert receipt['verified'] and receipt['members'][str(snap.relative_to(root))]==sha(snap)
chart=json.loads(snap.read_text())['chart'];assert chart['error'] is None and len(chart['result'])==1
item=chart['result'][0];meta=item['meta'];assert meta['symbol']=='SOXX' and meta['currency']=='USD' and meta['exchangeTimezoneName']=='America/New_York'
timestamps=item['timestamp'];quote=item['indicators']['quote'][0];dates=pd.to_datetime(timestamps,unit='s',utc=True).tz_convert('America/New_York').date
raw=pd.DataFrame({'date':pd.to_datetime(dates),'open':quote['open'],'high':quote['high'],'low':quote['low'],'close':quote['close'],'volume':quote['volume']}).set_index('date')
assert raw.index.is_unique and raw.index.is_monotonic_increasing
assert np.isfinite(raw[['open','high','low','close']]).all().all() and (raw[['open','high','low','close']]>0).all().all()
assert (raw['low']<=raw[['open','close','high']].min(axis=1)).all() and (raw['high']>=raw[['open','close','low']].max(axis=1)).all()
s=(root/src['official_NAV_raw']).read_text();s=re.sub(r'&(?!amp;|lt;|gt;|quot;|apos;|#[0-9]+;|#x[0-9A-Fa-f]+;)', '&amp;',s);xml=ET.fromstring(s);ns={'s':'urn:schemas-microsoft-com:office:spreadsheet'};key='{urn:schemas-microsoft-com:office:spreadsheet}Name';w=next(x for x in xml.findall('s:Worksheet',ns) if x.attrib.get(key)=='Historical')
nav={}
for row in w.findall('.//s:Row',ns)[1:]:
 vals=[c.findtext('s:Data',default='',namespaces=ns) for c in row.findall('s:Cell',ns)];d=pd.Timestamp(datetime.datetime.strptime(vals[0],'%b %d, %Y').date());v=float(vals[1]);assert d not in nav and math.isfinite(v) and v>0;nav[d]=v
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/src['Qlib_calendar_provider']),region='us',kernels=1)
start,end=plan['fixed_calendar_overlap'].split('..');cal=pd.DatetimeIndex(D.calendar(start_time=start,end_time=end,freq='day'));assert len(cal)==2205
miss_yahoo=cal.difference(raw.index);miss_nav=cal.difference(pd.DatetimeIndex(nav));valid=cal.intersection(raw.index).intersection(pd.DatetimeIndex(nav))
frame=pd.DataFrame({'Yahoo_quote_close':raw.loc[valid,'close'],'issuer_NAV':pd.Series(nav).loc[valid]});frame['price_to_NAV']=frame['Yahoo_quote_close']/frame['issuer_NAV'];frame['adjacent_log_ratio_change']=np.log(frame['price_to_NAV']).diff()
assert np.isfinite(frame[['Yahoo_quote_close','issuer_NAV','price_to_NAV']]).all().all()
stats=lambda x:{'n':int(len(x)),'median':float(x.median()),'p05':float(x.quantile(.05)),'p95':float(x.quantile(.95)),'min':float(x.min()),'max':float(x.max())}
years={str(y):stats(g['price_to_NAV']) for y,g in frame.groupby(frame.index.year)};events={k:len(v) for k,v in item.get('events',{}).items()}
report={'decision':'read_only_Yahoo_raw_quote_close_official_NAV_source_parity_no_factor','plan_sha256':sha(card/'plan.json'),'Yahoo_total_daily_rows':len(raw),'Qlib_overlap_calendar_rows':len(cal),'matched_rows':len(frame),'missing_Yahoo_Qlib_sessions':[str(d.date()) for d in miss_yahoo],'missing_NAV_Qlib_sessions':[str(d.date()) for d in miss_nav],'price_to_NAV_overall':stats(frame['price_to_NAV']),'annual_price_to_NAV':years,'largest_adjacent_abs_log_ratio_change':float(frame['adjacent_log_ratio_change'].abs().max()),'large_adjacent_log_ratio_change_gt_0p1':int((frame['adjacent_log_ratio_change'].abs()>0.1).sum()),'Yahoo_events_in_response':events,'current_revised_history_not_PIT':True,'historical_publication_time_unqualified':True,'model_fits':0,'native_accounts':0,'future_SOXX_outcomes_read':0}
static=pd.concat({'feature':frame},axis=1);static.index.name='datetime';static['instrument']='SOXX';static=static.set_index('instrument',append=True)
ds=DatasetH(handler=DataHandlerLP.from_df(static),segments={'source_overlap':(start,end)});assert len(ds.prepare('source_overlap',col_set=['feature']))==len(frame)
tracking=card/'qlib_recorder';assert not tracking.exists()
with R.start(experiment_name=plan['experiment'],uri=str(tracking)):
 rec=R.get_recorder();rid=rec.id;eid=str(rec.experiment_id);ds.config(dump_all=True,recursive=True);rec.save_objects(artifact_path='inputs',plan=plan,dataset=ds,source_hashes={'Yahoo':sha(snap),'issuer_NAV':sha(root/src['official_NAV_raw'])});rec.save_objects(artifact_path='analysis',source_frame=frame,report=report);rec.log_metrics(matched_rows=len(frame),model_fits=0,native_accounts=0)
(card/'result.json').write_text(json.dumps(report,indent=2)+'\n')
summary={'experiment':plan['experiment'],'decision':report['decision'],'objective_achieved':False,'plan_sha256':report['plan_sha256'],'source_archive':'nas_source_archive.json','recorder_id':rid,'experiment_id':eid,'recorder_uri':str(tracking),'result':'result.json','new_model_fits':0,'new_native_accounts':0,'artifact_policy':plan['artifact_policy'],'limitations':['Yahoo quote closes are current revised bars and not independently time-stamped first receipts.','Issuer NAV publication timing and revisions remain unqualified.','Parity only checks price basis; information/0bp Stage1 account not run.']}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
settings=_settings(root);binding=card/'nas_binding.json';binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':plan['experiment'],'ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n')
e=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=rid,experiment_id=eid)
with tempfile.TemporaryDirectory(prefix='soxx-nav-yahoo-') as tmp:
 archive=Path(tmp)/'recorder.tar.gz'
 with archive.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(e['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert run.returncode==0 and sha(archive)==e['sha256']
 with tarfile.open(archive,'r:gz') as z:assert any(m.name.endswith('/artifacts/analysis/report') for m in z.getmembers())
summary['nas_evidence']=e;summary['nas_readback']={'verified':True,'sha256':e['sha256']};(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
print(json.dumps({'rows':len(frame),'missing_Yahoo':len(miss_yahoo),'ratio':report['price_to_NAV_overall'],'annual_median':{k:v['median'] for k,v in years.items()},'recorder_id':rid},indent=2))


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[22962:MainThread](2026-10-23 07:47:40,445) INFO - qlib.Initialization - [config.py:453] - default_conf: client.


[22962:MainThread](2026-10-23 07:47:40,447) INFO - qlib.Initialization - [__init__.py:82] - qlib successfully initialized based on client settings.


[22962:MainThread](2026-10-23 07:47:40,447) INFO - qlib.Initialization - [__init__.py:84] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


[22962:MainThread](2026-10-23 07:47:40,477) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | Loading data Done


...

/Users/rziiiii/Library/Python/3.14/lib/python/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[22962:MainThread](2026-10-23 07:47:40,485) WARNING - qlib.workflow - [expm.py:230] - No valid experiment found. Create a new experiment with name stage1_soxx_nav_yahoo_raw_close_source_20261023.


[22962:MainThread](2026-10-23 07:47:40,486) INFO - qlib.workflow - [exp.py:258] - Experiment 421611483785789510 starts running ...


[22962:MainThread](2026-10-23 07:47:40,540) INFO - qlib.workflow - [recorder.py:345] - Recorder f2cfbeb58e004d079ac129eaeb447af6 starts running under Experiment 421611483785789510 ...


[22962:MainThread](2026-10-23 07:47:41,575) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | waiting `async_log` Done


{
  "rows": 2205,
  "missing_Yahoo": 0,
  "ratio": {
    "n": 2205,
    "median": 1.0000282052603318,
    "p05": 0.9990648173307141,
    "p95": 1.000958328750481,
    "min": 0.9958238548788834,
    "max": 1.0130285920401298
  },
  "annual_median": {
    "2018": 0.9999999930785609,
    "2019": 1.000096358724969,
    "2020": 0.9998485530675199,
    "2021": 1.0000464827664328,
    "2022": 1.0003025189980401,
    "2023": 1.000023602854434,
    "2024": 1.0000463724562638,
    "2025": 0.9999951990188921,
    "2026": 1.0000387721558097
  },
  "recorder_id": "f2cfbeb58e004d079ac129eaeb447af6"
}


# SOXX 原始收盘价与发行人 NAV：来源尺度对齐

冻结一次 Yahoo SOXX 日线请求，使用 `indicators.quote.close` 原始收盘价，与 iShares 官方历史 NAV 同日匹配；**没有使用 `adjclose`**。来源响应有 3,974 个日线行、64 个分红事件对象和 1 个拆分事件对象。按冻结 Qlib `D.calendar` 的 2018-01-02 至 2026-10-09 全部 2,205 个交易日，Yahoo 收盘与发行人 NAV 均有值，原生 DatasetH/Recorder 也为 2,205 行。

原始价格/NAV 比例在来源尺度上吻合：全期中位数 **1.000028**，5%–95% 分位 **0.999065–1.000958**；2018–2026 各年中位数都在 0.99985 至 1.00030，未出现相邻日绝对对数比例变化超过 0.1。此前冻结 Qlib close/NAV 比例从 0.927 平滑升至 0.999，故不能用那个调整后价格直接计算溢折价。这里的接近 1 只证明**当前两份历史序列的同日价格尺度可对齐**，不是预测信息或交易收益。

Yahoo 与发行人 NAV 都是本次获取的当前历史版本，尚未认证各交易日原始首次公布时刻、当时版本及修订；事件对象数量也不证明全部公司行动处理。来源原件和 Qlib Recorder 已分别归档 Yanxuan NAS、按 SHA-256 冷读，原生报告与 2,205 行 DatasetH 读回一致。0 模型拟合、0 账户、0 未来 SOXX 收益读取。

**决定：价格尺度必要门通过，历史时点门仍未通过；不形成可交易特征，不推广。** 下一步只能在明确标记“历史时点未认证”的条件下，事先冻结一个全区间信息检验，或先取得原始版本/发布时间证据；任何信息改善仍须完整 Stage1 0bp Tiered 自有账户及合同 v2 后续门。
